# 07e · 语言模型特征：ESM-2

## 这一章在整个项目里的位置

```
00–06 ✅ → [07 特征和基线] → 08 深度学习 → 09 全蛋白组预测 → 10 对外 API
              │
              ├─ 07a 设计 ✅   07b 新数据 ✅   07c 标签 ✅   07d 结构 ✅
              ├─ 07e 语言模型特征：ESM-2   ← 你在这里
              └─ 07f 特征表 + 第一个基线模型 + MLflow
```

## 背景：为什么需要这一章

07d 的结构特征回答的是"这个氨基酸在蛋白里处境如何"。还有一类同样重要的信息：**进化**。几亿年里，这个位置换过哪些氨基酸？经常变的位置，换了往往没事；从来不变的位置，换了多半出问题。

旧 QAFI 用 PSI-BLAST 搜同源序列、做多序列比对来算保守性：慢，依赖外部数据库，原始文件也没留下来。现在的标准做法是**蛋白质语言模型**：模型读过几千万条天然蛋白序列，自己学会了"哪个位置能换成什么"。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2、3 步：在本机用 ESM-2 算 TP53，再和 ProteinGym 的公开成绩对照（第一次要下载 2.6 GB 的模型）
   ├─ 第 4 步：把模型放进 raw（AWS），先让验货员认识模型文件（Terraform）
   └─ 第 5 步：看你的 Mac 后台计算的进度，上传结果；Databricks 的新流水线 language_model 装表
看结果：这里的输出；Databricks 网页的 Jobs 和 Catalog → qafi_dev → silver
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 蛋白质语言模型是什么 | 读过所有蛋白的校对员 |
| 2 | 在本机跑一次：TP53 | 先让校对员看一页熟悉的稿子 |
| 3 | 打分对不对：和 ProteinGym 对照 | 拿标准答案核对 |
| 4 | 模型也是数据：版本写死，进 raw | 进口零件要登记批号 |
| 5 | 全蛋白组：在你的 Mac 上算，Databricks 装表 | 搬家：东西自己拉过去，新家只管收货 |
| 6 | 看结果 | 分数和实验标签对一对 |

**花费：0**。**时间**：本机约 10 分钟（大部分是下载模型），另有一个后台全量计算（电脑空着时约 6 小时；你用电脑时它会自动让路、慢下来），不用你守着；Databricks 约 约 5 分钟。

---

## 准备

运行下面这一格：连上 Databricks 和 AWS，载入这一章的代码。**不用看代码。**

In [ ]:
import io, os, subprocess, sys, tempfile, urllib.request, zipfile
from pathlib import Path
import boto3
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from databricks.sdk import WorkspaceClient
from qafi_platform import ingest, lakehouse
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "databricks" / "src"))   # the code the Databricks job runs
from lakehouse import language_model as lm
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
aws = boto3.Session(profile_name="qafi", region_name="eu-central-1")
CATALOG, RAW, SOURCES = "qafi_dev", "qafi-186972156153-dev-raw", ROOT / "config/sources.toml"
print("Databricks:", w.config.host)

---

## 第 1 步 · 蛋白质语言模型是什么

### 在干什么
弄清楚 ESM-2 是什么，它给我们两样什么东西。

### 背景
比喻：**读过所有蛋白的校对员**。训练时，每条序列随机遮住 15% 的氨基酸，让模型猜被遮住的是什么。猜了几千万条序列以后，它就知道每个位置"通常是什么、换成什么还说得过去"。这种训练方式叫**掩码语言模型**（masked language model），和 BERT 读英文一样，只不过这里的"字母"是 20 种氨基酸。

它给我们两样东西：

| 产出 | 是什么 | 用在哪 |
|---|---|---|
| **零样本分数** | 每个位置 20 种氨基酸的概率。突变的分数 = log p(突变后) − log p(原来)。越负，模型越觉得"这不像天然蛋白"，突变越可能有害 | 07f 的特征 |
| **残基向量**（embedding） | 模型最后一层给每个残基的 1,280 个数字：模型对这个位置的"理解" | 08 号的神经网络 |

"**零样本**"（zero-shot）的意思：模型从没见过我们的实验标签，分数完全来自它读过的天然序列。

### 用了什么技术，为什么
- **ESM-2**（Meta，2023）：在 6,500 万条序列（UniRef50）上训练，大小从 800 万到 150 亿参数。我们用 **6.5 亿参数**（650M）这一档：ProteinGym 上它和 30 亿、150 亿的成绩差不多，但小得多
- **Hugging Face**：模型和代码最常用的发布平台。`transformers` 库一行就能载入，版本写死到具体的提交号（commit）
- **一次读完整条序列**（wild-type marginal）：读一遍，就得到每个位置所有 20 种氨基酸的概率。还有一种更准一点的读法，每个位置单独遮住读一次（masked marginal），但一条 400 个氨基酸的蛋白就要读 400 遍；全蛋白组 1,140 万个位置，读不起
- **超长蛋白**：ESM-2 一次最多读 1,022 个氨基酸。更长的蛋白切成互相重叠一半的窗口，每个残基取它**最居中**的那个窗口，和 07d 拼 AlphaFold 片段是同一个想法

### 公司里什么情况怎么做
蛋白质语言模型已经是变异效应预测和蛋白设计的标准特征：ESM1b 给人类所有可能的错义突变打过分，AlphaMissense 也用了类似的思路。先用别人训练好的模型做**特征**（这一章），以后再考虑**微调**（08c）。

### 面试怎么说
> "I use ESM-2 650M as a zero-shot feature: one forward pass per protein gives every residue's amino-acid probabilities, and a substitution's score is the log-likelihood ratio of mutant to wild type. Its last layer also gives a 1,280-dimensional embedding per residue for the deep-learning models."

---

## 第 2 步 · 在本机跑一次：TP53

### 在干什么
在自己电脑上用 ESM-2 读 TP53，看几个熟悉的突变得多少分。

### 背景
代码在 `databricks/src/lakehouse/language_model.py`，和 Databricks 上跑的是同一份：`lm.load()` 载入模型，`lm.read_proteins()` 读序列。你的 Mac 自带 GPU（Apple 芯片的 GPU，PyTorch 里叫 MPS），代码会自动用上它，读一个蛋白不到一秒；没有 GPU 的电脑用 CPU，慢十几倍。

**第一次运行要下载 2.6 GB 的模型**（存在 `~/.cache/huggingface`，以后不用再下）：

In [ ]:
model, tokenizer = lm.load()
tp53 = "".join(urllib.request.urlopen("https://rest.uniprot.org/uniprotkb/P04637.fasta").read().decode().splitlines()[1:])
((logprobs, vectors),) = lm.read_proteins([tp53], model, tokenizer, embeddings=True)
print(f"TP53: {len(tp53)} residues -> log-probabilities {logprobs.shape}, vectors {vectors.shape}")

def esm_score(mutation):   # "R175H": log p(H) - log p(R) at position 175
    wt, position, mt = mutation[0], int(mutation[1:-1]), mutation[-1]
    assert tp53[position - 1] == wt
    return logprobs[position - 1, lm.AMINO_ACIDS.index(mt)] - logprobs[position - 1, lm.AMINO_ACIDS.index(wt)]

every = logprobs - logprobs[np.arange(len(tp53)), [lm.AMINO_ACIDS.index(a) for a in tp53]][:, None]
print(f"median score of all {len(tp53) * 19:,} possible substitutions: {np.median(every[every != 0]):.2f}")
pd.DataFrame({"mutation": ["R175H", "G245S", "R248Q", "R249S", "R273H", "R282W", "P72R"],
              "known as": ["cancer hotspot"] * 6 + ["common polymorphism, benign"]}).assign(
    esm_score=lambda d: d["mutation"].map(esm_score).round(2))

### 结果怎么看
- TP53 有 393 个残基：每个残基 20 个概率，再加一个 1,280 维的向量
- 所有可能突变的分数**中位数约 −5.6**：先有这个参照，才知道一个分数算高算低
- 5 个癌症热点（R175H、G245S、R248Q、R273H、R282W）都在 **−9 到 −11**，远低于中位数：模型很确定这些位置不能换
- **P72R** 是人群里很常见的良性多态性，分数约 −6.1，和中位数差不多
- **R249S** 也是热点，分数却只有约 −6.6，单看分数分不出它和 P72R。07d 里 R249 是埋在蛋白内部的（RSA 0.14），结构特征能补上这一点

**一句话**：零样本分数能排出大致的好坏顺序，但单看一个数字下不了结论。这正是 07f 要把多种特征合在一起训练模型的原因。

### 面试怎么说
> "Zero-shot scores rank variants well but are not calibrated on their own: TP53 hotspots score around minus 10, yet R249S scores close to a benign polymorphism. That is why they are one feature among several, next to structure."

---

## 第 3 步 · 打分对不对：和 ProteinGym 对照

### 在干什么
确认我们的打分代码没写错。

### 背景
比喻：**拿标准答案核对**。ProteinGym 公布了 ESM-2 650M 在每个实验上的成绩：分数和实验结果的 **Spearman 相关系数**（只看排序是否一致，−1 到 1，越高越好）。我们用同样的模型、同样的序列算一遍，如果相关系数和公布的差不多，说明代码对了。

下载 ProteinGym 的实验数据（43 MB，放在临时文件夹，不进 raw）和公布的成绩表，挑 4 个人类蛋白的实验算一遍，其中 BRCA1 有 1,863 个氨基酸，要分 3 个窗口读，正好检查窗口拼接：

In [ ]:
folder = Path(tempfile.mkdtemp())
urllib.request.urlretrieve("https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.3/DMS_ProteinGym_substitutions.zip", folder / "dms.zip")
published = pd.read_csv("https://raw.githubusercontent.com/OATML-Markslab/ProteinGym/main/benchmarks/DMS_zero_shot/substitutions/Spearman/DMS_substitutions_Spearman_DMS_level.csv").set_index("DMS ID")
reference = pd.read_csv("https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.3/DMS_substitutions.csv").set_index("DMS_id")
archive = zipfile.ZipFile(folder / "dms.zip")
files = {name.rsplit("/", 1)[-1]: name for name in archive.namelist()}

rows = []
for assay in ["P53_HUMAN_Kotler_2018", "PTEN_HUMAN_Matreyek_2021", "CALM1_HUMAN_Weile_2017", "BRCA1_HUMAN_Findlay_2018"]:
    sequence = reference.loc[assay, "target_seq"]
    data = pd.read_csv(io.BytesIO(archive.read(files[reference.loc[assay, "DMS_filename"]])))
    data = data[~data["mutant"].str.contains(":")]   # single substitutions only
    ((scores, _),) = lm.read_proteins([sequence], model, tokenizer)
    wt, position, mt = data["mutant"].str[0], data["mutant"].str[1:-1].astype(int) - 1, data["mutant"].str[-1]
    ours = scores[position, mt.map(lm.AMINO_ACIDS.index)] - scores[position, wt.map(lm.AMINO_ACIDS.index)]
    rows.append({"assay": assay, "residues": len(sequence), "windows": len(lm.windows(len(sequence))),
                 "variants": len(data), "ours": spearmanr(ours, data["DMS_score"]).statistic,
                 "ProteinGym ESM2 650M": published.loc[assay, "ESM2 (650M)"]})
pd.DataFrame(rows).round(3)

### 结果怎么看
- 两列几乎一样，差别在 0.02 以内。我写这一章时核对了 12 个实验：平均 **0.459 对 0.465**
- 我们略低一点是正常的：ProteinGym 用的是每个位置单独遮住读的方法（第 1 步说的），更准，也贵几百倍
- BRCA1 分 3 个窗口读，成绩照样对得上：窗口拼接没问题

### 公司里什么情况怎么做
用别人的模型时，**先复现一个公开的数字**再往下做。复现不了，就是代码、版本或数据哪里不对，越早发现越省事。07a 复现旧论文的指标，也是这个道理。

### 面试怎么说
> "Before using the scores I reproduced ProteinGym's published ESM-2 650M Spearman correlations within 0.02 on twelve human assays, including long proteins read in windows."

---

## 第 4 步 · 模型也是数据：版本写死，进 raw

### 在干什么
把模型文件按我们进数据的老规矩，放进 S3 的 raw。

### 背景
比喻：**进口零件要登记批号**。模型的权重就是一个 2.6 GB 的文件，和 ClinVar、AlphaFold 一样是外部数据。它变了，所有特征都会变。所以：
- **版本写死**：`config/sources.toml` 里记下 Hugging Face 的提交号 `08e4846…`，不用"最新版"
- **走同一条路**：送货员（03b 的 Lambda）把 5 个文件从 Hugging Face 直接搬进 raw，验货员检查、开收货单；Databricks 再用提货单搬过去。Databricks 不需要能上 Hugging Face，以后 Hugging Face 上的东西改了、没了，我们也不受影响

模型权重是 `.safetensors` 格式，验货员原来不认识。给它加一条规则：**这种文件前 8 个字节是长度，第 9 个字节一定是 `{`**。这样，如果下载到的是一个报错网页，会被挡下来：

In [ ]:
from qafi_platform.raw_validator import SIGNATURES
print(SIGNATURES)
[s for s in ingest.load_sources(SOURCES) if s["dataset"] == "esm2-650m"]

### 执行
验货员的代码改了，用 Terraform 部署新版本（plan 应该是 **1 to change**：只更新验货员这一个 Lambda）：

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color")
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")

送货员把 5 个文件搬进 raw（2.6 GB，几分钟）：

In [ ]:
esm = [s for s in ingest.load_sources(SOURCES) if s["dataset"] == "esm2-650m"]
ingest.fetch_sources(aws.client("lambda"), aws.client("s3"), RAW, esm, timeout=900)

### 结果怎么看
- plan：`aws_lambda_function.raw_validator will be updated in-place`，`Plan: 0 to add, 1 to change, 0 to destroy.`
- 搬运：5 行 `accepted`，`model.safetensors` 2.61 GB。如果显示 `skipped ... (already in raw)`，说明已经搬过了

### 公司里什么情况怎么做
- 第三方模型和第三方数据一样管理：**版本写死、存一份在自己这里、记录来源**。生产任务不直接依赖公网
- 自己训练的模型放进**模型注册表**（09 号的 MLflow / Unity Catalog），那是另一回事

### 面试怎么说
> "Third-party model weights are treated as data: pinned to a Hugging Face commit, copied into the raw bucket with a validation receipt, and loaded from there, so the GPU job doesn't depend on the internet."

---

## 第 5 步 · 全蛋白组：在你的 Mac 上算，Databricks 装表

### 在干什么
给全蛋白组 20,652 个蛋白算语言模型特征，放进 silver 表。

### 背景
比喻：**换一台专门干这活的机器**。神经网络的计算几乎全是矩阵乘法，GPU 一次能做几千个，CPU 只能做几十个。全蛋白组要读 1,440 万个残基（长蛋白的窗口有重叠），实测：

| 在哪里算 | 每秒读多少个残基 | 全蛋白组要多久 |
|---|---|---|
| Mac 的 CPU | 57 | 约 71 小时 |
| Mac 的 GPU（M4） | 716 | 约 5.6 小时 |
| Databricks 的 GPU（A10） | 免费版用不了 | — |

07d 的结构计算用不上 GPU（全是查邻居、数个数）；这里正好相反。**什么活用什么机器**。

**Databricks 的 GPU 为什么用不了**：我先在 Databricks 上拿 200 个蛋白试跑。GPU 任务排了 27 分钟队，最后报错：

```
RESOURCE_EXHAUSTED: Cannot create max-num-nodes-gpu-1xa10 because it would exceed the configured limit of 0 ... [ai-compute:gpu-quota-denied]
```

意思是：这个免费版工作区的 A10 **配额是 0**。云上的资源都有配额，这种事只有真跑一下才知道。

所以计算放在你的 Mac 的 GPU 上，Databricks 只负责装表。比喻：**搬家**。东西自己开车拉过去（**回填**，backfill），新家只管收货、摆放：

```
你的 Mac（后台，两段）：
  ① 有标签的 457 个蛋白：概率 + 每个残基的向量  ──▶ data/esm2-650m/<版本>-labelled/
  ② 全蛋白组 20,652 个蛋白：概率               ──▶ data/esm2-650m/<版本>/
        │ 上传到 Volume
        ▼
language_model 流水线：
  load_language_model（普通 CPU）：装进三张 silver 表
  silver_language_model（GPU：A10）：只算还缺的。有 GPU 配额的工作区才能跑，免费版跳过
```

### 用了什么技术，为什么
- **回填用本机 GPU**：**同一份代码、同一个模型版本、同样的全精度**，结果和在云上的 GPU 算的一样
- **有标签的先算**：07f、08 号训练用的就是这 457 个蛋白（39 万个残基），先把它们算好，不用等全蛋白组
- **后台、低优先级、不睡眠**：`nohup` 让它在关掉终端后接着跑；`nice -n 19` 让它排在你所有程序后面；`caffeinate -i` 让电脑在它跑完之前不自动睡眠（合上盖子还是会睡，晚上让它跑就开着盖子、插着电）
- **内存紧张时让路**：16 GB 的 Mac 上，模型要占 3.6 GB。每算一批（50 个蛋白）之前，它先问 macOS 内存压力是不是"正常"；不正常就把模型从内存里卸掉，至少等 5 分钟再试。每批算完还歇一会儿，把 GPU 让给别的程序。你用电脑时它慢慢跑，电脑空着时全速跑
- **存成和表一样的格式**：`read_to_parquet` 每批存一组 parquet 文件，列和 silver 表一模一样。每批的"蛋白清单"最后才写，所以中途断了，重跑会接着算，不会重复
- **装表时只补缺的**：`load_language_model` 只装表里还没有的蛋白（或者序列、模型版本变了的），以及带来了向量的蛋白。两个文件夹先装哪个都一样；装两遍，第二遍什么都不做
- **GPU 任务留在图纸里**：`silver_language_model` 写了 `hardware_accelerator: GPU_1xA10` 和带 CUDA 的 AI 环境。换到有 GPU 配额的工作区，它就能接手以后的增量（UniProt 改了序列、新有了标签的蛋白），代码不用改
- **全精度（32 位）**：GPU 上常用半精度（16 位）提速。实测在 Mac 的 GPU 上半精度只快 12%，概率却最多差 0.2；全精度下 Mac 的 GPU 和 CPU 只差 0.0001。我们要的是**不管哪台机器算，特征都一样**，所以统一用全精度
- **增量**：一个蛋白只有在**序列变了、模型换了**，或者**新有了标签还没存向量**时才重读。判断序列变没变，用的是序列的 **MD5 指纹**
- **向量只存有标签的蛋白**：全蛋白组 1,140 万个残基 × 1,280 个数字，要 60 GB；08 号训练只用得上有标签的那些（约 2 GB），09 号预测全蛋白组时现算

### 执行
两段计算我已经在你的 Mac 上启动了，用的是下面两条命令（在终端里运行；万一断了，再运行一遍会接着算）：

```bash
# ① 有标签的蛋白，连向量一起
nohup caffeinate -i nice -n 19 env PYTHONPATH=databricks/src uv run python -m lakehouse.language_model \
    data/uniprot-human/2026_03/UP000005640_9606.fasta.gz data/esm2-650m/08e4846e537177426273712802403f7ba8261b6c-labelled \
    --only data/esm2-650m/labelled_proteins.txt --embeddings >> data/esm2-650m/08e4846e537177426273712802403f7ba8261b6c-labelled/run.log 2>&1 &
# ② 全蛋白组
nohup caffeinate -i nice -n 19 env PYTHONPATH=databricks/src uv run python -m lakehouse.language_model \
    data/uniprot-human/2026_03/UP000005640_9606.fasta.gz data/esm2-650m/08e4846e537177426273712802403f7ba8261b6c >> data/esm2-650m/08e4846e537177426273712802403f7ba8261b6c/run.log 2>&1 &
```

看进度：

In [ ]:
FOLDERS = [lm.REVISION + "-labelled", lm.REVISION]   # 有标签的蛋白（带向量），全蛋白组
for name in FOLDERS:
    folder = ROOT / "data/esm2-650m" / name
    read = pd.read_parquet(folder / "proteins", columns=["uniprot_id", "embedded"])
    print(f"{name}: {len(read):,} proteins read, {int(read['embedded'].sum()):,} with vectors")
    print("   last line of the log:", (folder / "run.log").read_text().strip().splitlines()[-1])

### 结果怎么看
- 第一个文件夹：**457** 个蛋白，457 个带向量
- 第二个文件夹：到 **20,652** 才算完
- 日志的最后一行：速度和大约还要多久；`memory is tight` 说明它正在让路

都读完以后，上传到 Databricks 的 Volume（带向量的约 1.9 GB，另一个约 1.1 GB）：

In [ ]:
VOLUME = f"/Volumes/{CATALOG}/raw/files/_backfill/esm2-650m"
for name in FOLDERS:
    run(f"databricks fs cp -r --overwrite -p qafi {ROOT / 'data/esm2-650m' / name} dbfs:{VOLUME}/{name}")

部署，然后只跑流水线里的装表任务 `load_language_model`（免费版没有 GPU，跳过 GPU 任务），两个文件夹各装一次：

In [ ]:
run("cd databricks && databricks bundle deploy")
for name in FOLDERS:
    run(f"cd databricks && databricks bundle run language_model --only load_language_model --params backfill={VOLUME}/{name}")
lakehouse.latest_run(w, "qafi language model")

### 结果怎么看
- 两次都写着 **0 proteins to add**：我已经装过了，所以这次什么都不用做。第一次装的时候，是 457 个（带向量），然后 20,195 个（有标签的那 457 个已经带着向量装过，跳过）
- `latest_run`：`load_language_model` 是 SUCCESS，另外两个任务是 SKIPPED（这次没让它们跑）

### 去网页上看
Databricks → **Jobs & Pipelines** → `qafi language model`：三个方框，GPU 任务排在装表任务后面；Catalog → `silver` → `residue_language_model`、`residue_embedding`、`language_model_proteins`。

### 公司里什么情况怎么做
- **先小规模试跑**：配额、权限、环境这类问题，只有真跑一下才知道。拿 200 个蛋白试，半小时就发现 GPU 配额是 0，而不是等全量任务失败
- **云上资源有配额**：GPU 尤其紧张，公司里要提前申请。申请不到时，计算可以放在别处，只要代码、模型版本、精度都一样，再装进同一张表
- **回填和增量分开**：第一次的大计算单独跑一次，用当时能用的算力；流水线只管以后的增量。写成和表一样的格式，装表时只补缺的、能重复跑
- **按工作选机器**：数据处理用 CPU，神经网络用 GPU，放在同一条流水线的不同任务里
- **开发在本地**：调试和小样本在自己电脑上跑（第 2、3 步和单元测试），云上的 GPU 只跑正式任务

### 面试怎么说
> "A 200-protein trial run showed that the Databricks Free Edition workspace has a GPU quota of zero. So the GPU stage stays in the pipeline for a workspace that has one, and the features were computed as a backfill on a laptop GPU, with the same code, model version and precision, saved in the silver tables' format and loaded by an idempotent CPU task.\"

---

## 第 6 步 · 看结果

### 在干什么
看全蛋白组装进了多少；装进表的数值和刚算的是否一样；零样本分数和我们的实验标签对得上多少。

In [ ]:
sql(w, f'''
SELECT model_version, count(*) AS proteins, sum(residues) AS residues,
       sum(CAST(embedded AS INT)) AS with_vectors, sum(CASE WHEN windows > 1 THEN 1 ELSE 0 END) AS read_in_windows
FROM {CATALOG}.silver.language_model_proteins GROUP BY model_version''')

### 结果怎么看
- **20,652** 个蛋白全部装进表，一共 **1,145.6 万**个残基
- `with_vectors`：**457**，就是有标签的那些蛋白，另存了每个残基 1,280 维的向量
- 2,317 个超过 1,022 个氨基酸的长蛋白是分窗口读的

表里的数值是后台任务算好、存成 parquet、上传、装进表以后的样子。和第 2 步在这个 notebook 里刚算的 TP53 比一比，看这一路有没有把数值弄变：

In [ ]:
import json
table = sql(w, f"SELECT position, logprobs FROM {CATALOG}.silver.residue_language_model WHERE uniprot_id = 'P04637' ORDER BY position")
stored = np.array([json.loads(x) for x in table["logprobs"]], dtype=float)   # the numbers arrive as text
print(f"largest difference in any log-probability: {np.abs(stored - logprobs).max():.6f}")

### 结果怎么看
- TP53 的 393 个残基、每个 20 个概率，最大差在 0.0001 以内（我核对时是 0.000001）：存成 parquet、上传、装表、再传回来，数值一路没被改变

最后，零样本分数和我们 07c 的实验标签对得上多少：每组实验算一个 Spearman 相关系数，再按来源取中位数：

In [ ]:
sql(w, f'''
WITH scored AS (
    SELECT l.dataset_id, CASE WHEN l.is_domainome THEN 'Domainome' ELSE l.source END AS source, l.label,
           element_at(r.logprobs, instr('{lm.AMINO_ACIDS}', l.alt_aa))
               - element_at(r.logprobs, instr('{lm.AMINO_ACIDS}', l.ref_aa)) AS esm_score
    FROM {CATALOG}.gold.fct_variant_label l
    JOIN {CATALOG}.silver.residue_language_model r ON r.uniprot_id = l.uniprot_id AND r.position = l.uniprot_position
), ranked AS (   -- Spearman = the correlation of the ranks; tied values share their average rank
    SELECT dataset_id, source,
           rank() OVER (PARTITION BY dataset_id ORDER BY esm_score)
               + (count(*) OVER (PARTITION BY dataset_id, esm_score) - 1) / 2.0 AS score_rank,
           rank() OVER (PARTITION BY dataset_id ORDER BY label)
               + (count(*) OVER (PARTITION BY dataset_id, label) - 1) / 2.0 AS label_rank
    FROM scored
), per_dataset AS (
    SELECT dataset_id, first(source) AS source, corr(score_rank, label_rank) AS spearman FROM ranked GROUP BY dataset_id
)
SELECT source, count(*) AS datasets, round(percentile(spearman, 0.5), 3) AS median_spearman,
       round(percentile(spearman, 0.1), 3) AS worst_tenth, round(percentile(spearman, 0.9), 3) AS best_tenth
FROM per_dataset GROUP BY source ORDER BY source''')

### 结果怎么看
| 来源 | 实验组数 | 中位数 | 最差的十分之一 | 最好的十分之一 |
|---|---|---|---|---|
| Domainome | 500 | 0.447 | 0.266 | 0.635 |
| 其他 MaveDB | 79 | 0.398 | 0.184 | 0.629 |
| ProteinGym 稳定性 | 13 | 0.471 | 0.222 | 0.631 |

- 零样本分数和实验结果的排序**中等程度一致**：中位数 0.40–0.47。模型从没见过这些标签，能做到这样已经不错，和第 3 步 ProteinGym 公布的 ESM-2 成绩是同一个水平
- 最差的十分之一只有 0.2 左右：有些实验测的是结合、表达量这类性质，语言模型只看序列，不一定看得出来
- 107 万个有标签的突变，**全部**都有语言模型特征

### 公司里什么情况怎么做
- 一个新特征上线前，看三件事：**覆盖了多少**（全蛋白组）、**数值稳不稳**（装表前后一致；Mac 的 GPU 和 CPU 也只差 0.0001）、**有没有信号**（和标签的相关性）
- 单个特征的相关系数只是起点。07f 会看它和结构特征**合在一起**能好多少

### 面试怎么说
> "Checks before a feature goes into training: coverage, numerical agreement between the stored values and a fresh computation, and its correlation with the labels per dataset."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)   # 第一次会自动整理 notebook 的格式
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 07e ESM-2 language-model features on a serverless GPU") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。这次 CI 要多装一个 PyTorch（只装 CPU 版，几十秒）。

---

## 本章小结

| 步骤 | 做了什么 | 为什么 |
|---|---|---|
| 1 概念 | ESM-2：读过 6,500 万条蛋白的掩码语言模型 | 用"天然序列里什么常见"代替比对算保守性 |
| 2 本机 | TP53：热点约 −10，中位数约 −5.6 | 分数能排顺序，单看一个数下不了结论 |
| 3 核对 | 12 个实验和 ProteinGym 公布的成绩差 0.02 以内 | 先复现公开数字，再往下做 |
| 4 进 raw | 模型版本写死，走送货员、验货员、提货单 | 模型也是数据，不依赖公网 |
| 5 全蛋白组 | 试跑发现 Databricks 的 GPU 配额是 0；Mac 的 GPU 回填，CPU 任务装表 | 先小规模试跑；同一份代码，结果一样 |
| 6 检查 | 覆盖率、装表前后数值一致、和标签的相关性 | 特征上线前的三项检查 |

**一句话**：全蛋白组每个残基都有了 20 种氨基酸的概率，每个可能的突变都能直接算出零样本分数；有标签的蛋白还存下了每个残基的向量，留给 08 号的神经网络。

## 面试题

1. **Why a protein language model instead of a multiple sequence alignment?**
   > "An alignment needs a homology search for every protein; a language model needs one forward pass and no database at prediction time, so it scales to the whole proteome with the same code. On ProteinGym the best alignment-based models are still slightly better, which is why conservation from alignments stays a candidate feature."

2. **What is a zero-shot variant score?**
   > "The model was never trained on our labels. It is the log-likelihood ratio of the mutant to the wild-type amino acid at that position, from the model's probabilities given the rest of the sequence."

3. **How do you make GPU batch inference efficient?**
   > "Sort sequences by length so batches need little padding, fill each batch up to a token budget that fits in memory, and save results batch by batch so a failure doesn't lose the run. Choose the precision on purpose: here half precision was only 12% faster but moved log-probabilities by up to 0.2, so features are computed in full precision and agree across machines."

**下一步 07f · 特征表 + 第一个基线模型**：把标签、结构特征、语言模型特征拼成一张表，用树模型训练第一个基线，用 MLflow 记录每一次实验。